In [ ]:
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
from fct_parallel import run_fct_plot as run_single_fct_plot
from plot_utils import DEFAULT_DPI, FIG_DIR, FONT_SIZE, TEXT_WIDTH, save_and_trim

try:
    from IPython.display import display
except ImportError:
    display = print


In [ ]:
def plot_fct_results(fct_results_df, variable_name, variable_values, labels, fct_metric, fig_name, xticks=None, xlim=None):
    """
    Plot FCT results with one subplot for each network, using logarithmic scales and custom ticks,
    ensuring the entire plot has a fixed width, and display a single legend.

    Args:
        fct_results_df (dict): Dictionary of DataFrames containing FCT data for each network.
        variable_name (str): Name of the variable being varied (e.g., "load" or "seed").
        variable_values (list): List of variable values.

    Returns:
        None
    """
    num_networks = len(fct_results_df)
    total_width = TEXT_WIDTH  # Fixed total width in inches
    height = LONG_FIG_RATIO * total_width  # Fixed height in inches

    fig, axes = plt.subplots(1, num_networks, figsize=(total_width, height), sharey=False, dpi=DEFAULT_DPI)

    if num_networks == 1:
        axes = [axes]  # Ensure axes is iterable when there is only one subplot

    # Define custom tick patterns
    xticks = XTICKS_HD if xticks is None else xticks
    first_yticks = YTICKS_HD_AVG if fct_metric=="avg" else (YTICKS_HD_P99 if fct_metric=="p99" else [])
    other_yticks = [0.4, 0.6, 0.8, 1, 1.2, 1.4, 1.6]

    all_handles = []  # To collect all legend handles
    all_labels = []   # To collect all legend labels

    for idx, (network_name, df) in enumerate(fct_results_df.items()):
        ax = axes[idx]

        # Shade background regions
        small_flows_label = f"Small flows"
        large_flows_label = f"Large flows"
        cutoff_label = f"Cut off\n(60 MB)" # Draw a vertical line at x = 60,000,000

        small_flows_patch = ax.axvspan(xmin=0, xmax=CUTOFF, facecolor="lightblue", alpha=DEFAULT_ALPHA, label=small_flows_label)
        large_flows_patch = ax.axvspan(xmin=CUTOFF, xmax=max(xticks), facecolor="lightgreen", alpha=DEFAULT_ALPHA, label=large_flows_label)
        vline = ax.axvline(x=CUTOFF, color="grey", linestyle="--", linewidth=0.75, label=cutoff_label)

        # Ensure these are added to the legend only once
        if small_flows_label not in all_labels:
            all_handles.append(small_flows_patch)
            all_labels.append(small_flows_label)
        if large_flows_label not in all_labels:
            all_handles.append(large_flows_patch)
            all_labels.append(large_flows_label)
        if cutoff_label not in all_labels:
            all_handles.append(vline)
            all_labels.append(cutoff_label)


        for variable in variable_values:
            column_name = f"{variable_name.capitalize()} {variable}"
            if column_name in df:
                # Plot and collect handles and labels
                line, = ax.plot(df["Size"], df[column_name], marker="o", markersize=1.5, linewidth=0.5, label=f"{variable_name.capitalize()} {round(float(variable))}%")
                if line.get_label() not in all_labels:  # Avoid duplicate labels
                    all_handles.append(line)
                    all_labels.append(line.get_label())

        

        # Set logarithmic scale
        if idx == 0:
            ax.set_yscale("log")
        ax.set_xscale("log")
        ax.set_xlim(xticks[0], xticks[-1])
        ax.set_xticks(xticks)
        ax.set_title(f"{labels[idx]}", fontsize=FONT_SIZE)


        # Customize Y-axis
        y_ticks = first_yticks if idx == 0 else other_yticks
        ax.set_yticks(y_ticks)
        ax.set_ylim(y_ticks[0], y_ticks[-1])
        if idx != 0:
            ax.yaxis.set_major_formatter(StrMethodFormatter("{x}"))

        

        # Add labels and grid
        ax.set_xlabel("Flow size (bytes)", fontsize=FONT_SIZE-1)
        ax.set_ylabel(f"{fct_metric} FCT (ms)" if idx == 0 else (f"Normalized {fct_metric} FCT" if idx == 1 else ""), fontsize=FONT_SIZE-1, labelpad=5)
        ax.tick_params(axis='both', labelsize=FONT_SIZE-1, length=2.5, width=0.5)
        ax.grid(True, linewidth=0.25)

        # Set subplot frame (spines) linewidth
        for spine in ax.spines.values():
            spine.set_linewidth(0.5)

    # Add a single legend
    fig.legend(
        handlelength=1.25,
        markerscale=1.0,
        handles=all_handles,
        labels=all_labels,
        loc="center right",  # Position legend at the right middle
        fontsize=FONT_SIZE-2,
        frameon=True
    )

    # Adjust layout to make space for the legend and reduce subplot spacing
    
    plt.tight_layout(rect=[0, 0, 0.92, 1])  # Adjust layout to make space for the legend
    
    w = 0.015
    for idx, ax in enumerate(axes):
    # Adjust the position of the subplot
        pos = ax.get_position()  # Get current position
        
        if idx == 0:  # Example adjustment for the second subplot
            ax.set_position([pos.x0, pos.y0, pos.width + w, pos.height])  # Shift subplot to the right
        elif idx == 1:
            ax.set_position([pos.x0 + 0.005, pos.y0, pos.width + w, pos.height])
        elif idx == 2:
            ax.set_position([pos.x0 - 0.02, pos.y0, pos.width + w, pos.height])

    save_and_trim(f"{FIG_DIR}/{fig_name}.png", dpi=DEFAULT_DPI)
    # Save the figure with equal cropping at the top and bottom
    plt.show()




In [ ]:
XTICKS_HD = [10**3, 10**4, 10**5, 10**6, 10**7, 10**8, 10**9]
XTICKS_DM = [10**2, 10**3, 10**4, 10**5, 10**6, 10**7, 10**8, 10**9]
YTICKS_HD_AVG = [10**-3, 10**-2, 10**-1, 10**0, 10**1, 10**2, 10**3]
YTICKS_HD_P99 = [10**-3, 10**-2, 10**-1, 10**0, 10**1, 10**2, 10**3, 10**4]
LONG_FIG_RATIO = 0.225
CUTOFF = 60_000_000
DEFAULT_ALPHA = 0.18

EXP_NAME = "FCT_uniform"
FILE_TEMPLATE = "../../results/FCT_uniform/log_{network}_HD_{variable}pload_seed={seed}.txt"
NETWORKS = ['cbb_final_108i', 'opera_ecmp', 'clos_prio']
LABELS = ['CBB-Net', 'Opera', '3:1 Fattree']
VARIABLE_NAME = "load"
VARIABLE_VALUES = ['2.00', '4.00', '6.00', '8.00', '10.00']
SEEDS = ['1', '2', '3', '4', '5']
XTICKS = XTICKS_HD
XLIM = (XTICKS_HD[0], XTICKS_HD[-1])


def run_fct_plot(fct_metric):
    """Compute averaged FCT data and plot one metric."""
    return run_single_fct_plot(
        fct_metric,
        exp_name=f"{EXP_NAME}_HD",
        file_template=FILE_TEMPLATE,
        networks=NETWORKS,
        variable_name=VARIABLE_NAME,
        variable_values=VARIABLE_VALUES,
        seeds=SEEDS,
        labels=LABELS,
        xticks=XTICKS,
        xlim=XLIM,
        plot_fct_results_func=plot_fct_results,
        display_func=None,
        display_network=None,
    )


In [ ]:
avg_fct_results_df = run_fct_plot("avg")


In [ ]:
p99_fct_results_df = run_fct_plot("p99")
